# Sentinel-2 piltide lõikamine metsaeraldiste bounding boxi järgi

Lõikab **kõik** `get_data.py` defineeritud alad (train ja test) välja **kõigist** alla laetud Sentinel-2 L2A toodetest.
Iga ala × toode kombinatsioon annab eraldi GeoTIFF-i: `sentinel2_clips/{ala}/{toode}.tif`.

Sisend: `data/metsaregister/` (`get_data.py` väljund) ja `data/sentinel2_downloads/*.zip` (allalaadimise märkmiku väljund).
Sõltuvused: `pip install rasterio geopandas numpy pandas`

In [ ]:
import csv
import json
import math
import re
import zipfile
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import transform_bounds
from rasterio.windows import Window

## Seaded

In [ ]:
STANDS_DIR = Path("data/metsaregister")           # get_data.py väljund
S2_DIR     = Path("data/sentinel2_downloads")     # allalaetud .SAFE.zip failid
OUTPUT_DIR = Path("data/sentinel2_clips")

SPLITS = {"train", "test"}        # millised alad lõigata (kõik)
BBOX_SOURCE = "stands"            # "stands" = valitud eraldiste bbox, "area" = kogu 10x10 km ruut
BUFFER_M = 0                      # lisapuhver bboxile meetrites
MAX_CLOUD = None                  # nt 20 -> jäta välja lõiked, mille pilvisus on üle 20 %
RESAMPLING = Resampling.bilinear  # 20 m ribade resampling 10 m peale (SCL on alati nearest)
OVERWRITE = False

SRC_CRS = "EPSG:3301"
# (riba, algne eraldusvõime m)
BANDS = [("B02", 10), ("B03", 10), ("B04", 10), ("B08", 10),
         ("B05", 20), ("B06", 20), ("B07", 20), ("B8A", 20),
         ("B11", 20), ("B12", 20), ("SCL", 20)]
CLOUD_CLASSES = (3, 8, 9, 10)     # vari, pilv, õhuke pilv
BAND_RE = re.compile(r"IMG_DATA/R(\d+)m/.*_(B\d[0-9A]|SCL)_\d+m\.jp2$")
NAME_RE = re.compile(r"^S2[ABC]_MSIL2A_(\d{8})T\d{6}_(N\d{4})_R\d{3}_(T\w{5})_")

## Abifunktsioonid

In [ ]:
def band_paths(zip_path):
    """{(riba, resolutsioon): /vsizip/ tee} ZIP-i seest, lahti pakkimata."""
    paths = {}
    with zipfile.ZipFile(zip_path) as z:
        for entry in z.namelist():
            m = BAND_RE.search(entry)
            if m:
                paths[(m.group(2), int(m.group(1)))] = f"/vsizip/{zip_path.resolve().as_posix()}/{entry}"
    return paths


def read_band(path, win, resolution, resampling):
    """Loe riba 10 m aknale; 20 m ribad resamplitakse."""
    col, row, w, h = win
    with rasterio.open(path) as src:
        if resolution == 10:
            return src.read(1, window=Window(col, row, w, h))
        return src.read(1, window=Window(col // 2, row // 2, w // 2, h // 2),
                        out_shape=(h, w), resampling=resampling)


def clip_product(zip_path, bounds_src, out_path):
    """Lõika üks toode. Tagasta (metaandmed | None, staatus)."""
    paths = band_paths(zip_path)
    missing = [b for b, _ in BANDS if not any(k[0] == b for k in paths)]
    if missing:
        return None, f"puuduvad ribad: {missing}"

    def pick(band, want):
        res = want if (band, want) in paths else min(r for b, r in paths if b == band)
        return paths[(band, res)], res

    with rasterio.open(pick("B02", 10)[0]) as ref:
        xmin, ymin, xmax, ymax = transform_bounds(SRC_CRS, ref.crs, *bounds_src, densify_pts=21)
        t = ref.transform
        # joondus 20 m ruudustikule, et 10 m ja 20 m ribad kattuksid
        col0 = math.floor((xmin - t.c) / 20) * 2
        col1 = math.ceil((xmax - t.c) / 20) * 2
        row0 = math.floor((t.f - ymax) / 20) * 2
        row1 = math.ceil((t.f - ymin) / 20) * 2
        if col0 < 0 or row0 < 0 or col1 > ref.width or row1 > ref.height:
            return None, "bbox ei mahu tile'i sisse"
        win = (col0, row0, col1 - col0, row1 - row0)
        transform = ref.window_transform(Window(*win))
        crs = ref.crs

    names, stack = [], []
    for band, want in BANDS:
        path, res = pick(band, want)
        method = Resampling.nearest if band == "SCL" else RESAMPLING
        stack.append(read_band(path, win, res, method))
        names.append(band)
    data = np.stack(stack)

    valid = data[0] > 0                      # B02 == 0 -> nodata
    if not valid.any():
        return None, "ainult nodata"
    scl = data[names.index("SCL")]
    cloud_frac = float(np.isin(scl, CLOUD_CLASSES)[valid].mean())
    if MAX_CLOUD is not None and cloud_frac * 100 > MAX_CLOUD:
        return None, f"pilvisus {cloud_frac:.0%} > {MAX_CLOUD}%"

    out_path.parent.mkdir(parents=True, exist_ok=True)
    profile = dict(driver="GTiff", dtype="uint16", count=len(names), crs=crs,
                   transform=transform, height=data.shape[1], width=data.shape[2],
                   compress="deflate", tiled=True, blockxsize=256, blockysize=256, nodata=0)
    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(data)
        for i, name in enumerate(names, start=1):
            dst.set_band_description(i, name)
        dst.update_tags(SOURCE_PRODUCT=zip_path.name, BANDS=",".join(names))
    return {"valid_fraction": round(float(valid.mean()), 4),
            "cloud_fraction": round(cloud_frac, 4),
            "width": data.shape[2], "height": data.shape[1], "crs": str(crs)}, "ok"

## Alade ja toodete laadimine

In [ ]:
summary = json.loads((STANDS_DIR / "summary.json").read_text(encoding="utf-8"))
areas = [a for a in summary["areas"] if a["split"] in SPLITS]
zips = sorted(S2_DIR.glob("*.zip"))
print(f"Alad: {[(a['name'], a['split']) for a in areas]}")
print(f"Sentinel-2 tooteid: {len(zips)}")

## Lõikamine (kõik alad × kõik tooted)

In [ ]:
manifest = []
for area in areas:
    name = area["name"]
    stands_file = STANDS_DIR / f"{name}.geojson"
    gdf = gpd.read_file(stands_file)
    if BBOX_SOURCE == "stands":
        if gdf.empty:
            print(f"{name}: eraldisi pole, jätan vahele.")
            continue
        b = tuple(gdf.to_crs(SRC_CRS).total_bounds)
    else:
        b = tuple(area["bbox"])
    bounds = (b[0] - BUFFER_M, b[1] - BUFFER_M, b[2] + BUFFER_M, b[3] + BUFFER_M)
    print(f"\n== {name} ({area['split']}), bbox {[round(v) for v in bounds]} ==")

    for zip_path in zips:
        product = zip_path.name.removesuffix(".zip").removesuffix(".SAFE")
        m = NAME_RE.match(product)
        out_path = OUTPUT_DIR / name / f"{product}.tif"
        row = {"area": name, "split": area["split"], "product": product,
               "tile": m.group(3) if m else "", "date": m.group(1) if m else "",
               "baseline": m.group(2) if m else "", "n_stands": len(gdf),
               "bbox_3301": list(bounds), "stands_file": str(stands_file),
               "clip_file": "", "status": ""}
        try:
            if out_path.exists() and not OVERWRITE:
                row.update(status="olemas", clip_file=str(out_path))
            else:
                info, status = clip_product(zip_path, bounds, out_path)
                if info:
                    row.update(info, status="ok", clip_file=str(out_path))
                else:
                    row["status"] = status
        except Exception as exc:       # rikkis ZIP vms - jätka järgmisega
            row["status"] = f"viga: {exc}"
        print(f"  {product}: {row['status']}")
        manifest.append(row)

## Manifest ja kokkuvõte

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
df = pd.DataFrame(manifest)
df.to_csv(OUTPUT_DIR / "manifest.csv", index=False)

df["ok"] = df["status"].isin(["ok", "olemas"])
print(f"Lõikeid kokku: {df['ok'].sum()} / {len(df)}")
display(df.groupby(["area", "split"])["ok"].sum().rename("lõikeid").reset_index())
display(df[~df["ok"]].groupby(["area", "tile", "status"]).size().rename("tooteid").reset_index())

## Kontroll: millised tile'id katavad millised alad

In [ ]:
display(df[df["ok"]].pivot_table(index="area", columns="tile", values="product", aggfunc="count", fill_value=0))

## Eelvaade (valikuline)

In [ ]:
import matplotlib.pyplot as plt

ok = df[df["ok"]].sort_values("cloud_fraction")
areas_ok = ok["area"].unique()
fig, axes = plt.subplots(1, len(areas_ok), figsize=(5 * len(areas_ok), 5), squeeze=False)
for ax, area in zip(axes[0], areas_ok):
    best = ok[ok["area"] == area].iloc[0]          # vähima pilvisusega lõige
    with rasterio.open(best["clip_file"]) as src:
        rgb = src.read([3, 2, 1]).astype("float32")   # B04, B03, B02
    rgb = np.clip((rgb - 1000) / 3000, 0, 1).transpose(1, 2, 0)
    ax.imshow(rgb)
    ax.set_title(f"{area}\n{best['date']} {best['tile']} (pilv {best['cloud_fraction']:.0%})")
    ax.axis("off")
plt.tight_layout()
plt.show()